In [1]:
from pathlib import Path
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import pandas as pd
import numpy as np 

In [16]:
ROOT = Path("/home/MOBODY")
RUN_DIR = ROOT / "runs_friction_shift_sweep_seed1"

In [ ]:
tb_dirs = sorted(set(p.parent for p in RUN_DIR.rglob("events.out.tfevents*")))

rows = []

for tb_dir in tb_dirs:
    event_files = sorted(tb_dir.glob("events.out.tfevents*"), key=lambda p: p.stat().st_mtime)
    event_path = event_files[-1]  # 최신 event만 사용

    ea = EventAccumulator(str(event_path))
    ea.Reload()
    tags = ea.Tags()["scalars"]

    def get_last(tag):
        if tag not in tags:
            return np.nan, np.nan
        vals = ea.Scalars(tag)
        if len(vals) == 0:
            return np.nan, np.nan
        last = vals[-1].value
        last5 = np.mean([v.value for v in vals[-5:]])
        return last, last5

    src_last, src_mean = get_last("test/source_return")
    trg_last, trg_mean = get_last("test/target_return")
    norm_last, norm_mean = get_last("test/target_normalized_score")

    p = str(event_path)

    if "DARA" in p:
        policy = "DARA"
    elif "IQL" in p:
        policy = "IQL"
    elif "MOBODY" in p:
        policy = "MOBODY"
    else:
        policy = "UNKNOWN"

    if "walker2d-friction" in p:
        env = "walker2d-friction"
    elif "hopper-friction" in p:
        env = "hopper-friction"
    elif "halfcheetah-friction" in p:
        env = "halfcheetah-friction"
    else:
        env = "UNKNOWN"

    rows.append({
        "tb_dir": str(tb_dir),
        "event_path": str(event_path),
        "policy": policy,
        "env": env,
        "src_return_last": src_last,
        "src_return_mean": src_mean,
        "trg_return_last": trg_last,
        "trg_return_mean": trg_mean,
        "norm_score_last": norm_last,
        "norm_score_mean": norm_mean,
    })

df_latest = pd.DataFrame(rows)
df_latest.sort_values(["env", "policy"], inplace=True)

In [ ]:
df_latest

,tb_dir,event_path,policy,env,src_return_last,src_return_mean,trg_return_last,trg_return_mean,norm_score_last,norm_score_mean
0,/home/MOBODY/runs_friction_shift_sweep_seed1/D...,/home/MOBODY/runs_friction_shift_sweep_seed1/D...,DARA,halfcheetah-friction,2971.293701,2716.856323,14347.979492,14231.856445,34.848259,34.571623
1,/home/MOBODY/runs_friction_shift_sweep_seed1/D...,/home/MOBODY/runs_friction_shift_sweep_seed1/D...,DARA,halfcheetah-friction,5032.360840,5041.057813,4713.259766,4758.041211,65.382675,65.969034
2,/home/MOBODY/runs_friction_shift_sweep_seed1/D...,/home/MOBODY/runs_friction_shift_sweep_seed1/D...,DARA,halfcheetah-friction,5300.183594,5307.741016,5292.214844,5204.182617,48.303768,47.540672
3,/home/MOBODY/runs_friction_shift_sweep_seed1/D...,/home/MOBODY/runs_friction_shift_sweep_seed1/D...,DARA,halfcheetah-friction,5154.381836,5136.910840,3509.696777,4113.695117,36.164639,41.928255
12,/home/MOBODY/runs_friction_shift_sweep_seed1/I...,/home/MOBODY/runs_friction_shift_sweep_seed1/I...,IQL,halfcheetah-friction,4949.335449,4881.170117,5064.447754,5395.769482,12.732358,13.521656
13,/home/MOBODY/runs_friction_shift_sweep_seed1/I...,/home/MOBODY/runs_friction_shift_sweep_seed1/I...,IQL,halfcheetah-friction,5092.747070,5125.943164,4966.823242,4856.103027,68.702766,67.253025
14,/home/MOBODY/runs_friction_shift_sweep_seed1/I...,/home/MOBODY/runs_friction_shift_sweep_seed1/I...,IQL,halfcheetah-friction,5259.012695,5283.171191,5334.846680,5278.246582,48.673317,48.182687
15,/home/MOBODY/runs_friction_shift_sweep_seed1/I...,/home/MOBODY/runs_friction_shift_sweep_seed1/I...,IQL,halfcheetah-friction,5178.025879,5191.324512,4491.707520,3782.432080,45.535416,38.767200
24,/home/MOBODY/runs_friction_shift_sweep_seed1/M...,/home/MOBODY/runs_friction_shift_sweep_seed1/M...,MOBODY,halfcheetah-friction,4869.214355,4867.219531,3494.678223,3308.109961,8.992739,8.548282
25,/home/MOBODY/runs_friction_shift_sweep_seed1/M...,/home/MOBODY/runs_friction_shift_sweep_seed1/M...,MOBODY,halfcheetah-friction,4296.455566,4391.977344,3788.685791,4121.850879,53.276566,57.638938
